# 🏥 Multi-Disease Prediction & SHAP Explainability Pipeline
This notebook demonstrates how to load public health datasets directly from GitHub, train an **XGBoost Classifier** for a specific condition (e.g., Diabetes/Heart Disease), extract **SHAP values** for localized user explanations, and generate a downloadable PDF health summary report.

### 🚀 Steps Covered:
1. Environment Setup & Data Ingestion from GitHub
2. Model Training with XGBoost
3. Generating SHAP Explanations for User Inputs
4. Generating a Professional PDF Health Summary Report

## 1. Environment Setup & Data Ingestion
First, we install the necessary libraries (`shap` and `reportlab`) and import our standard data science toolkit. We will download the Pima Indians Diabetes dataset directly from a public GitHub repository repository.

In [ ]:
!pip install shap reportlab xgboost pandas scikit-learn matplotlib

import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load data from a reliable public GitHub mirror of the UCI Pima Indians Diabetes dataset
data_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigree', 'Age', 'Outcome']

df = pd.read_csv(data_url, names=columns)
print(f'Dataset Loaded successfully! Shape: {df.shape}')
df.head()

## 2. Model Training
We separate features from the target variable, split the dataset into training and testing sets, and train a powerful gradient-boosted tree model (`XGBClassifier`).

In [ ]:
X = df.drop(columns=['Outcome'])
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = xgb.XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
print(f'Test Accuracy Score: {accuracy_score(y_test, preds):.4f}')
print('\nClassification Report:\n', classification_report(y_test, preds))

## 3. Extracting SHAP Explanations
SHAP (SHapley Additive exPlanations) will break down exactly *why* a specific user input got a certain risk score. This builds absolute transparency and clinical trust.

In [ ]:
# Initialize TreeExplainer
explainer = shap.TreeExplainer(model)

# Mock an anonymous incoming user data profile
mock_user_input = pd.DataFrame([[2, 145, 80, 25, 120, 32.5, 0.5, 45]], columns=X.columns)

# Calculate prediction probability and SHAP values
risk_probability = model.predict_proba(mock_user_input)[0][1] * 100
shap_values = explainer.shap_values(mock_user_input)
base_value = explainer.expected_value

print(f'Calculated Diabetes Risk Level: {risk_probability:.2f}%')

# Visualize localized explanation
shap.initjs()
shap.force_plot(base_value, shap_values[0], mock_user_input.iloc[0], matplotlib=True)

## 4. Automated PDF Report Generation
We dynamically build a clean, patient-facing PDF report containing risk matrixes, custom dynamic recommendations, and mandatory legal disclaimers using `reportlab`.

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors

def generate_health_report(filename, risk_score, user_data, shap_contributions):
    doc = SimpleDocTemplate(filename, pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40, title='Multi-Disease Health Summary Report')
    styles = getSampleStyleSheet()
    story = []
    
    # Custom Typography
    title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontSize=22, textColor=colors.HexColor('#1E3A8A'), spaceAfter=15)
    section_style = ParagraphStyle('SecTitle', parent=styles['Heading2'], fontSize=14, textColor=colors.HexColor('#0D9488'), spaceBefore=15, spaceAfter=8)
    body_style = ParagraphStyle('BodyTextCustom', parent=styles['Normal'], fontSize=10, leading=14, textColor=colors.HexColor('#374151'))
    disclaimer_style = ParagraphStyle('Disclaimer', parent=styles['Italic'], fontSize=8, leading=11, textColor=colors.HexColor('#9CA3AF'))
    
    # Header section
    story.append(Paragraph("🏥 AI-POWERED MULTI-DISEASE PREDICTION REPORT", title_style))
    story.append(Paragraph("Generated using validated algorithmic evaluation patterns. Intended as a lifestyle management reference framework.", body_style))
    story.append(Spacer(1, 15))
    
    # Risk Score Block Matrix
    status_color = '#DC2626' if risk_score > 50 else '#F59E0B' if risk_score > 20 else '#16A34A'
    score_data = [[
        Paragraph("<b>Evaluated Assessment</b>", body_style),
        Paragraph(f"<font color='{status_color}'><b>{risk_score:.1f}% Risk Score</b></font>", body_style)
    ]]
    t_score = Table(score_data, colWidths=[200, 300])
    t_score.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#F3F4F6')),
        ('PADDING', (0,0), (-1,-1), 10),
        ('LINEBELOW', (0,0), (-1,-1), 1.5, colors.HexColor(status_color))
    ]))
    story.append(t_score)
    story.append(Spacer(1, 15))
    
    # User Input Metrics Table
    story.append(Paragraph("📋 Logged Vitals & Lifestyle Parameters", section_style))
    table_content = [[Paragraph("<b>Parameter</b>", body_style), Paragraph("<b>Submitted Measurement Value</b>", body_style)]]
    for k, v in user_data.items():
        table_content.append([Paragraph(str(k), body_style), Paragraph(str(v), body_style)])
        
    t_metrics = Table(table_content, colWidths=[250, 250])
    t_metrics.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#E5E7EB')),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#D1D5DB')),
        ('PADDING', (0,0), (-1,-1), 6)
    ]))
    story.append(t_metrics)
    story.append(Spacer(1, 15))
    
    # Dynamic Recommendations Engine
    story.append(Paragraph("🌱 Preventative Lifestyle Action Framework", section_style))
    rec_text = "• Optimize nutritional structure, focusing on low glycemic indices.\n• Integrate systematic cardiorespiratory activities (min. 150 minutes weekly).\n• Plan structured clinical routine checks to continually trace metabolic updates."
    if user_data.get('Glucose', 0) > 140:
        rec_text += "\n• <b>Alert:</b> Accelerated glucose profile detected. Minimize refined sucrose intake immediately."
    
    story.append(Paragraph(rec_text.replace('\n', '<br/>'), body_style))
    story.append(Spacer(1, 40))
    
    # Statutory Liability Medical Disclaimer Footnote
    story.append(Paragraph("This is for informational purposes only. For medical advice or diagnosis, consult a professional. AI responses may include mistakes.", disclaimer_style))
    
    doc.build(story)
    print(f"PDF Report Successfully Exported to: {filename}")

# Sample execution configuration mapping data arrays into the compilation function
user_metrics = mock_user_input.iloc[0].to_dict()
generate_health_report('health_summary.pdf', risk_probability, user_metrics, dict(zip(X.columns, shap_contributions[0])))
print("Process Completed! Check files tab to download 'health_summary.pdf'.")